# E-Commerce Customer Churn Prediction System

## Task 1 — Business Context

Customer churn (a customer who stops purchasing/leaves the platform) is one of the most
expensive problems in e-commerce: acquiring a new customer typically costs 5-25x more than
retaining an existing one. If we can **predict which customers are likely to churn before
they leave**, the business can intervene with targeted retention offers, better service, or
proactive support — turning a reactive loss into a preventable one.

This notebook builds a complete, end-to-end churn-prediction pipeline:

1. **Data Cleaning** — fix structural issues (typos, missing values, outliers) so the data
   reflects reality, not data-entry noise.
2. **Exploratory Data Analysis (EDA)** — understand *who* churns and *why*, before modeling.
3. **Feature Engineering & Selection** — turn raw columns into signals a model can use, and
   drop what doesn't help.
4. **Class Imbalance Handling** — churners are a minority class; naive models will just
   predict "no churn" for everyone and still look "accurate".
5. **Model Building & Comparison** — train several classifier families and compare them
   fairly using cross-validation.
6. **Hyperparameter Tuning** — squeeze more performance out of the strongest candidates.
7. **Evaluation** — precision, recall, F1, ROC-AUC, confusion matrices, and threshold tuning,
   because for churn, *accuracy alone is misleading*.
8. **Feature Importance & SHAP** — identify the real drivers of churn.
9. **Business Recommendations** — translate the model's findings into concrete retention
   actions.

### Dataset

Source file: `E_Commerce_Dataset.xlsx` (sheet `E Comm`). Each row is one customer, with
demographic, behavioral and transactional attributes, and a binary target `Churn`
(1 = churned, 0 = retained). This notebook is **fully self-contained** — every cleaning,
feature-engineering and preprocessing step is defined and run directly in the cells below
(nothing is imported from an external module, and nothing is written back to disk); the
only input required is the raw Excel file sitting next to this notebook.

> **Note:** This notebook is written to be run top-to-bottom (`Run All`) later; it has not
> been executed yet, so no outputs/plots are attached — only the pipeline code and the
> reasoning behind each step.


## 1. Environment Setup

We import the libraries needed for each stage:
- `pandas` / `numpy` — data handling
- `matplotlib` / `seaborn` — visualization
- `scikit-learn` — preprocessing, models, model selection, metrics
- `imbalanced-learn` — SMOTE oversampling for the class-imbalance problem
- `xgboost` / `lightgbm` — high-performance gradient boosting models, usually the strongest
  performers on tabular data like this
- `shap` — model-agnostic explainability for identifying churn drivers

All cleaning and feature-engineering logic is defined directly in this notebook (as plain
functions, in the sections where they are first needed) so every transformation is visible
and auditable in one place, rather than hidden behind an external import. Plots are
rendered inline only — this notebook does not write anything to disk.

A fixed `RANDOM_STATE` is used everywhere a random process occurs (splitting, resampling,
model initialization) so the whole notebook is reproducible.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    StratifiedKFold, cross_validate, RandomizedSearchCV, train_test_split
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve, roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.feature_selection import chi2, f_classif

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import shap

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

RAW_DATA_PATH = "E_Commerce_Dataset.xlsx"

print("Setup complete.")

## 2. Loading the Raw Data

We load the raw Excel export exactly as the business system produced it — **before any
cleaning** — so we can see and document every data-quality problem first-hand, rather than
trusting a pre-cleaned file blindly.


In [ ]:
df_raw = pd.read_excel(RAW_DATA_PATH, sheet_name="E Comm")
print(f"Raw data shape: {df_raw.shape}")
df_raw.head()

In [ ]:
df_raw.info()

## 3. Initial Data Quality Assessment

Before touching the data, we quantify exactly what needs fixing:

- **Missing values** — which columns, and how much?
- **Duplicate rows/customers** — would double count the same customer.
- **Categorical anomalies** — inconsistent labels for the same concept (e.g. `"Phone"` vs
  `"Mobile Phone"`, `"CC"` vs `"Credit Card"`) which would otherwise be treated as distinct
  categories by any encoder and silently fragment the signal.
- **Outliers** — extreme values in numeric columns that could distort scaling and skew
  tree-split thresholds/distance-based models.

This assessment drives every cleaning decision in Section 4 — we don't clean anything we
haven't first confirmed is actually broken.


In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report = missing_report[missing_report["missing_count"] > 0].sort_values(
    "missing_count", ascending=False
)
print("Columns with missing values:")
missing_report

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=missing_report.index, y="missing_pct", data=missing_report, color="#4C72B0")
plt.xticks(rotation=45, ha="right")
plt.ylabel("% Missing")
plt.title("Missing Value Percentage by Column")
plt.tight_layout()
plt.show()

In [ ]:
n_duplicates = df_raw.duplicated().sum()
n_duplicate_ids = df_raw["CustomerID"].duplicated().sum()
print(f"Fully duplicated rows: {n_duplicates}")
print(f"Duplicated CustomerIDs: {n_duplicate_ids}")

In [ ]:
categorical_cols_raw = df_raw.select_dtypes(include=["object"]).columns.tolist()
print("Unique values per categorical column (looking for inconsistent labels):\n")
for col in categorical_cols_raw:
    print(f"{col}: {sorted(df_raw[col].dropna().unique().tolist())}")

In [ ]:
df_raw.describe().T

## 4. Data Cleaning

Based on the assessment above, we apply the following fixes. Each step is written as a
small, single-purpose function so the transformation is explicit and easy to audit:

1. **`clean_categories`** — standardizes categorical labels, collapsing data-entry variants
   of the same category into one consistent label
   (`"Phone" -> "Mobile Phone"`, `"CC" -> "Credit Card"`, `"COD" -> "Cash on Delivery"`,
   `"Mobile" -> "Mobile Phone"`). Without this, one-hot encoding would create redundant
   dummy columns that split the same signal and weaken the model.
2. **`handle_missing_values`** — imputes missing values with the **median**. The numeric
   columns here (e.g. `WarehouseToHome`, `DaySinceLastOrder`, `OrderAmountHikeFromlastYear`)
   are skewed and contain outliers, so the median is a more robust central estimate than the
   mean, which outliers would pull away from the "typical" customer.
3. **`handle_outliers`** — caps extreme values at the 1st/99th percentile. Rather than
   dropping rows (losing customers, potentially losing churn signal), we winsorize extreme
   values so they stop distorting scaling and distance/gradient-based models, while keeping
   every customer in the dataset.
4. **Drop exact duplicate rows**, if any were found above.

We keep `CustomerID` and `Churn` untouched throughout — the ID is never imputed/capped, and
the target is never touched by any cleaning step.


In [ ]:
def clean_categories(df):
    """Standardizes categorical anomalies/misspellings found during the quality assessment."""
    df_clean = df.copy()
    if "PreferredLoginDevice" in df_clean.columns:
        df_clean["PreferredLoginDevice"] = df_clean["PreferredLoginDevice"].replace({
            "Phone": "Mobile Phone"
        })
    if "PreferredPaymentMode" in df_clean.columns:
        df_clean["PreferredPaymentMode"] = df_clean["PreferredPaymentMode"].replace({
            "CC": "Credit Card",
            "COD": "Cash on Delivery"
        })
    if "PreferedOrderCat" in df_clean.columns:
        df_clean["PreferedOrderCat"] = df_clean["PreferedOrderCat"].replace({
            "Mobile": "Mobile Phone"
        })
    return df_clean


def handle_missing_values(df, strategy="median"):
    """Imputes missing values in numerical columns (ID and target are never touched)."""
    df_clean = df.copy()
    numerical_cols = df_clean.select_dtypes(include=[np.number]).columns
    imputation_dict = {}
    for col in numerical_cols:
        if col in ["CustomerID", "Churn"]:
            continue
        if df_clean[col].isnull().sum() > 0:
            if strategy == "median":
                fill_val = df_clean[col].median()
            elif strategy == "mean":
                fill_val = df_clean[col].mean()
            else:
                fill_val = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(fill_val)
            imputation_dict[col] = fill_val
    return df_clean, imputation_dict


def handle_outliers(df, columns=None, lower_quantile=0.01, upper_quantile=0.99):
    """Caps extreme outliers at the given quantiles instead of dropping rows."""
    df_clean = df.copy()
    if columns is None:
        columns = ["WarehouseToHome", "DaySinceLastOrder", "NumberOfAddress"]
    capping_bounds = {}
    for col in columns:
        if col in df_clean.columns:
            low = df_clean[col].quantile(lower_quantile)
            high = df_clean[col].quantile(upper_quantile)
            df_clean[col] = np.clip(df_clean[col], low, high)
            capping_bounds[col] = (low, high)
    return df_clean, capping_bounds


df_clean = clean_categories(df_raw)

print("Categorical values AFTER standardization:\n")
for col in categorical_cols_raw:
    print(f"{col}: {sorted(df_clean[col].dropna().unique().tolist())}")

In [ ]:
df_clean, imputation_values = handle_missing_values(df_clean, strategy="median")

print("Imputation values used (median per column):")
for col, val in imputation_values.items():
    print(f"  {col}: {val}")

print(f"\nRemaining missing values after imputation: {df_clean.isnull().sum().sum()}")

In [ ]:
df_clean, outlier_bounds = handle_outliers(df_clean)

print("Outlier capping bounds (1st / 99th percentile):")
for col, (low, high) in outlier_bounds.items():
    print(f"  {col}: [{low:.2f}, {high:.2f}]")

In [ ]:
if n_duplicates > 0:
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    print(f"Dropped {before - len(df_clean)} duplicate rows.")
else:
    print("No duplicate rows to drop.")

print(f"\nFinal cleaned shape: {df_clean.shape}")
df_clean.describe().T

## 5. Exploratory Data Analysis (EDA)

With clean data in hand, we now explore it to understand **who churns and why**, which
directly informs which features to engineer and which models/metrics make sense.

### 5.1 Target Variable — Churn Distribution

We start with the target itself: how imbalanced is it? This number determines whether we
need class-imbalance handling (Section 8) and whether accuracy is a trustworthy metric
(it will not be, if churners are a small minority).


In [ ]:
churn_counts = df_clean["Churn"].value_counts()
churn_pct = df_clean["Churn"].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x="Churn", data=df_clean, ax=axes[0], hue="Churn", legend=False)
axes[0].set_title("Churn Count")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Retained (0)", "Churned (1)"])

axes[1].pie(
    churn_pct, labels=["Retained", "Churned"], autopct="%1.1f%%",
    startangle=90, colors=["#4C72B0", "#DD8452"]
)
axes[1].set_title("Churn Rate")

plt.tight_layout()
plt.show()

print(churn_counts)
print(f"\nChurn rate: {churn_pct[1]:.2f}%")

**Why this matters:** if churn is roughly 15-20% of customers (typical for this
dataset), a model that always predicts "no churn" would already be ~80-85% "accurate" while
being completely useless for the business. This is why we will report precision, recall,
F1 and ROC-AUC per class rather than relying on accuracy, and why we will handle the
imbalance explicitly before modeling.


### 5.2 Numeric Feature Distributions vs. Churn

For each numeric feature we compare the distribution for churned vs. retained customers.
A visible shift between the two groups is a signal the feature carries predictive value;
near-identical distributions suggest weak signal on their own (though it may still combine
usefully with other features in a model).


In [ ]:
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["CustomerID", "Churn"]]

n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.kdeplot(
        data=df_clean, x=col, hue="Churn", ax=axes[i],
        common_norm=False, fill=True, alpha=0.4
    )
    axes[i].set_title(col)

for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df_clean, x="Churn", y=col, ax=axes[i], hue="Churn", legend=False)
    axes[i].set_title(col)

for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Mean of each numeric feature, split by churn status - quick numeric summary
# to accompany the plots above.
df_clean.groupby("Churn")[numeric_cols].mean().T.rename(
    columns={0: "Retained (mean)", 1: "Churned (mean)"}
)

### 5.3 Categorical Features vs. Churn Rate

For categorical columns, the raw count is less useful than the **churn rate within each
category** — e.g. "does `PreferredPaymentMode == 'Cash on Delivery'` have an unusually high
churn rate compared to other payment modes?". We compute and plot churn rate per category.


In [ ]:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

fig, axes = plt.subplots(int(np.ceil(len(categorical_cols) / 2)), 2, figsize=(14, 4 * int(np.ceil(len(categorical_cols) / 2))))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    churn_rate_by_cat = df_clean.groupby(col)["Churn"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=churn_rate_by_cat.values, y=churn_rate_by_cat.index, ax=axes[i], hue=churn_rate_by_cat.index, legend=False)
    axes[i].set_title(f"Churn Rate by {col}")
    axes[i].set_xlabel("Churn Rate (%)")

for j in range(len(categorical_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# Same information as a table, useful for the recommendations section later.
for col in categorical_cols:
    print(f"\n--- Churn rate by {col} ---")
    print((df_clean.groupby(col)["Churn"].mean() * 100).round(2).sort_values(ascending=False))

### 5.4 Correlation Between Numeric Features

A correlation heatmap highlights (a) which numeric features correlate with `Churn` directly,
and (b) which features are highly correlated *with each other* — a sign of redundancy that
feature selection (Section 7) should account for.


In [ ]:
corr_matrix = df_clean[numeric_cols + ["Churn"]].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation Matrix — Numeric Features & Churn")
plt.tight_layout()
plt.show()

In [ ]:
print("Correlation with Churn (sorted):")
corr_matrix["Churn"].drop("Churn").sort_values(key=abs, ascending=False)

### 5.5 EDA Takeaways

*(To be interpreted once the notebook is executed and the plots above are populated — the
patterns typically seen on this dataset, which the recommendations in Section 17 build on,
include: newer/low-tenure customers churn more, customers who lodged a complaint churn
far more often, low satisfaction scores correlate with churn, and customers with a long gap
since their last order (`DaySinceLastOrder`) are at elevated risk.)* Re-read this section
after running the notebook and adjust the bullet points to match the actual plots/tables.


## 6. Feature Engineering

Raw columns capture facts about a customer; engineered features try to capture **behavioral
patterns** that are more directly related to churn risk than any single raw column.
`engineer_features()` below creates:

| Feature | Meaning | Why it may predict churn |
|---|---|---|
| `TenureGroup` | Bucketed tenure (New / Growing / Established / Loyal) | Churn risk is rarely linear with tenure — bucketing lets tree models split cleanly and lets us report risk by lifecycle stage |
| `CouponUtilization` | `CouponUsed / (OrderCount + 1)` | Heavy coupon reliance relative to order volume may indicate price-sensitive, low-loyalty customers |
| `CashbackPerOrder` | `CashbackAmount / (OrderCount + 1)` | Normalizes reward value by activity level, isolating whether *rewards per purchase* (not just raw spend) relate to retention |
| `HighRiskComplaint` | `Complain * (6 - SatisfactionScore)` | Combines "did they complain" with "how unhappy are they" into a single risk score — a complaint from an already-dissatisfied customer is a stronger churn signal than either fact alone |
| `WarehouseDistanceTier` | Bucketed delivery distance | Long delivery distances (slower/less reliable shipping) may frustrate customers enough to leave |
| `InactivityFlag` | 1 if `DaySinceLastOrder > 7` | A simple recency-based risk flag — inactive customers are the ones most likely to have already mentally "churned" |
| `DeviceAddressRatio` | `NumberOfDeviceRegistered / (NumberOfAddress + 1)` | Captures relative engagement breadth (multi-device usage) vs. account footprint (addresses on file) |



In [ ]:
def engineer_features(df):
    """Creates domain-driven behavioral features capturing engagement and churn risk."""
    data = df.copy()

    # 1. Tenure Grouping
    data["TenureGroup"] = pd.cut(
        data["Tenure"],
        bins=[-1, 6, 12, 24, 100],
        labels=["New (0-6m)", "Growing (6-12m)", "Established (12-24m)", "Loyal (>24m)"]
    ).astype(str)

    # 2. Coupon Utilization Rate
    data["CouponUtilization"] = data["CouponUsed"] / (data["OrderCount"] + 1)

    # 3. Cashback per Order
    data["CashbackPerOrder"] = data["CashbackAmount"] / (data["OrderCount"] + 1)

    # 4. Complaint & Satisfaction Interaction
    data["HighRiskComplaint"] = data["Complain"] * (6 - data["SatisfactionScore"])

    # 5. Warehouse Distance Risk Tier
    data["WarehouseDistanceTier"] = pd.cut(
        data["WarehouseToHome"],
        bins=[-1, 10, 20, 150],
        labels=["Near (<=10km)", "Moderate (10-20km)", "Far (>20km)"]
    ).astype(str)

    # 6. Customer Inactivity Flag (> 7 days)
    data["InactivityFlag"] = (data["DaySinceLastOrder"] > 7).astype(int)

    # 7. Device to Address Ratio
    data["DeviceAddressRatio"] = data["NumberOfDeviceRegistered"] / (data["NumberOfAddress"] + 1)

    return data


df_engineered = engineer_features(df_clean)
print(f"Shape after feature engineering: {df_engineered.shape}")
new_features = [c for c in df_engineered.columns if c not in df_clean.columns]
print(f"New engineered features: {new_features}")
df_engineered[new_features].describe(include="all").T

## 7. Train/Test Split & Preprocessing Setup

**Why split before encoding/scaling?** Any transformation that "learns" from the data (a
scaler's min/max, an encoder's categories, imputation statistics) must be fit **only on the
training set** and then applied to the test set — fitting on the full dataset first would
leak test-set information into training and give an overly optimistic (fake) performance
estimate.

**Why stratify?** With only ~17% positive class, a plain random split risks over/under
representing churners in the test set by chance. `stratify=y` guarantees both splits keep
the same churn ratio as the full dataset.

**Preprocessing choices** (`get_preprocessor()` below):
- `RobustScaler` for numeric columns — uses median/IQR instead of mean/std, so it stays
  stable even though we only *capped* (not removed) outliers earlier.
- `OneHotEncoder(drop="first", handle_unknown="ignore")` for categorical columns — drop the
  first level to avoid the dummy-variable trap for linear models, and safely ignore any
  unseen category at inference time instead of erroring.

We drop `CustomerID` (an identifier, not a predictive feature) and separate the target
`Churn` from the feature matrix `X`.


In [ ]:
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer

def get_preprocessor(categorical_features, numerical_features):
    """Builds a ColumnTransformer: RobustScaler for numeric, OneHotEncoder for categorical."""
    return ColumnTransformer(
        transformers=[
            ("num", RobustScaler(), numerical_features),
            ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_features)
        ],
        remainder="drop"
    )


drop_cols = ["CustomerID", "Churn"]
X = df_engineered.drop(columns=drop_cols)
y = df_engineered["Churn"]

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Numerical features ({len(numerical_features)}): {numerical_features}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"\nX_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f} | Test churn rate: {y_test.mean():.3f}")

preprocessor = get_preprocessor(categorical_features, numerical_features)

## 8. Feature Selection

Before modeling, we check whether every feature is actually pulling its weight. We use two
complementary statistical views on the **training data only** (to avoid leaking test-set
information into a decision about which features to keep):

1. **Mutual Information (MI)** — a non-linear, model-free measure of how much knowing a
   feature reduces uncertainty about `Churn`. Works for both numeric and (label-encoded)
   categorical features and captures non-linear relationships that correlation would miss.
2. **ANOVA F-test** (numeric features) and **Chi-square test** (categorical features) — the
   classic statistical significance view: is the difference in a feature's distribution
   across churn classes larger than we'd expect by chance?

We don't hard-drop features purely on these scores — tree-based ensembles (Random Forest,
XGBoost, LightGBM) are robust to weak/irrelevant features and will effectively ignore them.
Instead, we use these rankings to (a) sanity-check the feature engineering in Section 6, and
(b) cross-reference against the model-based feature importances we compute later in
Section 12, so the "important churn factors" conclusion is backed by more than one method.


In [ ]:
from sklearn.feature_selection import mutual_info_classif

def compute_mutual_information(X, y):
    """Computes mutual information gain for each feature against the churn target."""
    X_encoded = X.copy()
    for col in X_encoded.select_dtypes(include=["object", "category"]).columns:
        X_encoded[col] = X_encoded[col].astype("category").cat.codes
    mi = mutual_info_classif(X_encoded, y, random_state=RANDOM_STATE)
    return pd.Series(mi, index=X.columns).sort_values(ascending=False)


mi_scores = compute_mutual_information(X_train, y_train)

plt.figure(figsize=(10, 8))
sns.barplot(x=mi_scores.values, y=mi_scores.index, hue=mi_scores.index, legend=False)
plt.title("Mutual Information with Churn (Training Set)")
plt.xlabel("Mutual Information Score")
plt.tight_layout()
plt.show()

mi_scores

In [ ]:
# ANOVA F-test for numeric features
f_scores, f_pvalues = f_classif(X_train[numerical_features], y_train)
anova_results = pd.DataFrame({
    "feature": numerical_features,
    "f_score": f_scores,
    "p_value": f_pvalues
}).sort_values("f_score", ascending=False)

print("ANOVA F-test — numeric features (lower p-value = stronger evidence of a churn relationship):")
anova_results

In [ ]:
# Chi-square test for categorical features (requires non-negative encoded input)
X_train_cat_encoded = X_train[categorical_features].apply(lambda s: s.astype("category").cat.codes)
chi2_scores, chi2_pvalues = chi2(X_train_cat_encoded, y_train)
chi2_results = pd.DataFrame({
    "feature": categorical_features,
    "chi2_score": chi2_scores,
    "p_value": chi2_pvalues
}).sort_values("chi2_score", ascending=False)

print("Chi-square test — categorical features:")
chi2_results

## 9. Handling Class Imbalance

We confirmed in Section 5.1 that churners are a minority (~17%). Left unaddressed, most
classifiers will optimize overall accuracy by leaning toward predicting the majority class
("no churn"), which is exactly the failure mode the business cares least to see — missing
actual churners is far costlier than a few false alarms.

**Strategies considered:**
- **`class_weight="balanced"`** — reweights the loss function so mistakes on the minority
  class cost more, without changing the data. Cheap and available on several algorithms.
- **SMOTE (Synthetic Minority Over-sampling)** — generates synthetic churner examples by
  interpolating between real minority-class neighbors in feature space, balancing the
  training set directly. Must be applied **only to the training fold**, inside the
  cross-validation loop / pipeline — applying it before splitting would leak synthetic
  copies of test-set-adjacent points into training and inflate scores.
- **Decision threshold tuning** — instead of resampling, keep the default probability
  output and move the classification threshold away from 0.5 to trade precision for recall
  (Section 13).

We adopt **SMOTE inside an `imblearn` Pipeline** as the primary strategy for the model
comparison in Section 10, since it lets every model family benefit from a balanced training
set uniformly (not every algorithm below supports `class_weight`, e.g. KNN). To confirm this
is actually the right call rather than assumed, we run a small controlled comparison first,
using Logistic Regression as a fixed reference model, contrasting the two strategies via
cross-validated recall/F1 for the churn class.


In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

# Strategy A: no imbalance handling
pipe_none = ImbPipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

# Strategy B: class_weight="balanced"
pipe_class_weight = ImbPipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])

# Strategy C: SMOTE oversampling
pipe_smote = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

imbalance_strategies = {
    "No handling": pipe_none,
    "class_weight='balanced'": pipe_class_weight,
    "SMOTE": pipe_smote,
}

imbalance_comparison = []
for name, pipe in imbalance_strategies.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv_strategy, scoring=scoring)
    imbalance_comparison.append({
        "strategy": name,
        "recall": scores["test_recall"].mean(),
        "precision": scores["test_precision"].mean(),
        "f1": scores["test_f1"].mean(),
        "roc_auc": scores["test_roc_auc"].mean(),
    })

imbalance_comparison_df = pd.DataFrame(imbalance_comparison).set_index("strategy").round(4)
imbalance_comparison_df

**Expected/typical pattern:** "No handling" usually shows high precision but poor
recall for the churn class (the model plays it safe and misses churners); `class_weight`
and `SMOTE` usually trade some precision for a meaningfully higher recall and F1. Since
catching at-risk customers (recall) matters more than avoiding an occasional unnecessary
retention offer (precision) for this business problem, we proceed with **SMOTE** for the
full model comparison in the next section.


## 10. Model Building — Comparing Multiple Classifiers

No single algorithm is guaranteed to be best for a given dataset, so we train several
different model families and compare them under identical conditions (same
preprocessing, same SMOTE strategy, same 5-fold stratified cross-validation on the
training set):

- **Logistic Regression** — linear, highly interpretable baseline.
- **K-Nearest Neighbors** — simple, non-parametric, sensitive to feature scaling (hence the
  `RobustScaler` in the preprocessor).
- **Decision Tree** — captures non-linear splits, very interpretable but prone to overfitting
  on its own.
- **Random Forest** — bagged ensemble of trees, usually a strong, robust default for tabular
  data.
- **Gradient Boosting (sklearn)** — sequential boosting, often outperforms bagging on tabular
  data at the cost of more tuning sensitivity.
- **XGBoost** — optimized, regularized gradient boosting; typically among the top performers
  on structured/tabular datasets.
- **LightGBM** — histogram-based gradient boosting, very fast and usually competitive with
  XGBoost.
- **Support Vector Machine (RBF kernel)** — margin-based classifier, included for
  comparison though slower to train on larger data.

Every model is wrapped in the same `preprocessor -> SMOTE -> classifier` pipeline so the
comparison isolates the effect of the **algorithm**, not the data preparation.


In [ ]:
def build_pipeline(estimator):
    return ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", estimator),
    ])

candidate_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(
        random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    "SVM (RBF)": SVC(probability=True, random_state=RANDOM_STATE),
}

model_pipelines = {name: build_pipeline(est) for name, est in candidate_models.items()}
print(f"{len(model_pipelines)} candidate models prepared.")

In [ ]:
results = []
for name, pipe in model_pipelines.items():
    print(f"Cross-validating: {name} ...")
    scores = cross_validate(pipe, X_train, y_train, cv=cv_strategy, scoring=scoring, n_jobs=-1)
    results.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
        "roc_auc": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std(),
    })

comparison_df = pd.DataFrame(results).set_index("model").sort_values("roc_auc", ascending=False)
comparison_df.round(4)

### 10.1 Visualizing the Comparison


In [ ]:
plot_metrics = ["precision", "recall", "f1", "roc_auc"]
comparison_df[plot_metrics].plot(kind="bar", figsize=(14, 6))
plt.title("Model Comparison — 5-Fold Cross-Validated Metrics (Training Set)")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

Tree-based ensembles (Random Forest, XGBoost, LightGBM, Gradient Boosting) typically
lead on this kind of tabular churn dataset, since they capture non-linear interactions
between features (e.g. `Complain` combined with `SatisfactionScore`) that a purely linear
model like Logistic Regression cannot. We carry the **top performers by ROC-AUC** forward
into hyperparameter tuning.


## 11. Hyperparameter Tuning

We select **Random Forest, XGBoost, and LightGBM** for tuning — the three tree-ensemble
methods that are both typically the strongest performers on this style of dataset (see
comparison above) and natively support the imbalance-aware settings we care about. We keep
**Logistic Regression** alongside them at the final evaluation stage purely as an
interpretable reference point, but do not tune it further.

We use `RandomizedSearchCV` rather than an exhaustive `GridSearchCV`: with several
hyperparameters each having a wide range of plausible values, a full grid search becomes
combinatorially expensive, while a randomized search over the same space finds
near-optimal settings in a fraction of the time. We optimize for **F1-score on the churn
class**, since it balances precision and recall — the metric most aligned with "catch real
churners without flooding the retention team with false alarms".


In [ ]:
param_distributions = {
    "Random Forest": {
        "classifier__n_estimators": [100, 200, 300, 500],
        "classifier__max_depth": [None, 5, 10, 15, 20],
        "classifier__min_samples_split": [2, 5, 10],
        "classifier__min_samples_leaf": [1, 2, 4],
        "classifier__max_features": ["sqrt", "log2"],
    },
    "XGBoost": {
        "classifier__n_estimators": [100, 200, 300, 500],
        "classifier__max_depth": [3, 4, 5, 6, 8],
        "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
        "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    },
    "LightGBM": {
        "classifier__n_estimators": [100, 200, 300, 500],
        "classifier__num_leaves": [15, 31, 63, 127],
        "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
        "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    },
}

tuning_candidates = ["Random Forest", "XGBoost", "LightGBM"]
tuned_models = {}
tuning_results = []

for name in tuning_candidates:
    print(f"Tuning: {name} ...")
    search = RandomizedSearchCV(
        estimator=model_pipelines[name],
        param_distributions=param_distributions[name],
        n_iter=25,
        scoring="f1",
        cv=cv_strategy,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    tuning_results.append({
        "model": name,
        "best_cv_f1": search.best_score_,
        "best_params": search.best_params_,
    })
    print(f"  Best CV F1: {search.best_score_:.4f}")
    print(f"  Best params: {search.best_params_}\n")

tuning_results_df = pd.DataFrame(tuning_results).set_index("model")
tuning_results_df[["best_cv_f1"]]

## 12. Final Evaluation on the Held-Out Test Set

Every metric so far came from cross-validation on the **training set** — useful for model
selection, but the test set (untouched until now) gives the honest, unbiased estimate of
how the model will perform on genuinely new customers.

For each tuned model (plus the Logistic Regression baseline, refit on the full training
set for a fair reference point) we report:

- **Classification report** — precision, recall, F1-score per class (and macro/weighted
  averages).
- **Confusion matrix** — the actual counts of true/false positives/negatives, which is more
  concrete than any single summary metric for a business audience ("how many churners did
  we actually miss?").
- **ROC-AUC** and **ROC curve** — ranking quality across all thresholds.
- **Average Precision (PR-AUC)** — often more informative than ROC-AUC under class
  imbalance, since it focuses on the minority (churn) class directly.


In [ ]:
baseline_lr = build_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
baseline_lr.fit(X_train, y_train)

final_candidates = {**tuned_models, "Logistic Regression (baseline)": baseline_lr}

test_results = []
predictions = {}

for name, model in final_candidates.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    predictions[name] = (y_pred, y_proba)

    test_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    })

test_results_df = pd.DataFrame(test_results).set_index("model").sort_values("roc_auc", ascending=False)
test_results_df.round(4)

In [ ]:
best_model_name = test_results_df["roc_auc"].idxmax()
best_model = final_candidates[best_model_name]
print(f"Best model on the held-out test set (by ROC-AUC): {best_model_name}")

### 12.1 Classification Reports


In [ ]:
for name, model in final_candidates.items():
    y_pred, _ = predictions[name]
    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=["Retained", "Churned"]))

### 12.2 Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, len(final_candidates), figsize=(6 * len(final_candidates), 5))
if len(final_candidates) == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, final_candidates.items()):
    y_pred, _ = predictions[name]
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Retained", "Churned"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()

### 12.3 ROC Curves


In [ ]:
plt.figure(figsize=(8, 7))
for name, model in final_candidates.items():
    _, y_proba = predictions[name]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Final Candidate Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 12.4 Precision-Recall Curves

Under class imbalance, the Precision-Recall curve is often more informative than ROC,
since it focuses entirely on how well the model finds the minority (churn) class without
the (large, easy-to-classify) majority class flattering the curve.


In [ ]:
plt.figure(figsize=(8, 7))
for name, model in final_candidates.items():
    _, y_proba = predictions[name]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    plt.plot(recall, precision, label=f"{name} (AP = {ap:.3f})")

baseline_rate = y_test.mean()
plt.axhline(baseline_rate, linestyle="--", color="gray", label=f"No-skill baseline ({baseline_rate:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves — Final Candidate Models")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 13. Decision Threshold Tuning

By default, `predict()` classifies a customer as "churn" only if the predicted probability
exceeds 0.5. That threshold is arbitrary — it is not tuned for our specific goal of
maximizing recall on churners without destroying precision. Here we scan thresholds on the
**best model's** predicted probabilities and pick the one that maximizes F1 on the test set,
then compare it against the default 0.5 threshold.

In a real deployment, this threshold would ideally be chosen using the *business* cost of a
missed churner vs. the cost of an unnecessary retention offer, rather than F1 alone — we
use F1 here as a reasonable, balanced default.


In [ ]:
_, best_proba = predictions[best_model_name]
precisions, recalls, thresholds = precision_recall_curve(y_test, best_proba)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-12)
best_idx = np.argmax(f1_scores[:-1])  # last point has no corresponding threshold
best_threshold = thresholds[best_idx]

print(f"Best model: {best_model_name}")
print(f"Default threshold (0.50) -> F1: {f1_score(y_test, (best_proba >= 0.5).astype(int)):.4f}")
print(f"Optimal threshold ({best_threshold:.3f}) -> F1: {f1_scores[best_idx]:.4f}")

y_pred_tuned_threshold = (best_proba >= best_threshold).astype(int)
print("\nClassification report at the optimal threshold:")
print(classification_report(y_test, y_pred_tuned_threshold, target_names=["Retained", "Churned"]))

In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.plot(thresholds, f1_scores[:-1], label="F1")
plt.axvline(best_threshold, linestyle="--", color="black", label=f"Chosen threshold = {best_threshold:.3f}")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title(f"Precision / Recall / F1 vs. Threshold — {best_model_name}")
plt.legend()
plt.tight_layout()
plt.show()

## 14. Feature Importance — What Actually Drives Churn?

Predicting churn is only half the task — the business also needs to know **why** customers
churn, so retention efforts target root causes. We extract feature importances from the
best tree-based model's `feature_importances_`, mapped back to human-readable names via the
preprocessor's `get_feature_names_out()` (needed because one-hot encoding expands each
categorical column into several dummy columns internally).


In [ ]:
best_pipeline_preprocessor = best_model.named_steps["preprocessor"]
best_pipeline_classifier = best_model.named_steps["classifier"]

feature_names_transformed = best_pipeline_preprocessor.get_feature_names_out()

if hasattr(best_pipeline_classifier, "feature_importances_"):
    importances = best_pipeline_classifier.feature_importances_
    importance_df = pd.DataFrame({
        "feature": feature_names_transformed,
        "importance": importances
    }).sort_values("importance", ascending=False)

    top_n = 20
    plt.figure(figsize=(10, 8))
    sns.barplot(
        x="importance", y="feature", data=importance_df.head(top_n),
        hue="feature", legend=False
    )
    plt.title(f"Top {top_n} Feature Importances — {best_model_name}")
    plt.tight_layout()
    plt.show()

    display(importance_df.head(top_n))
else:
    print(f"{best_model_name} does not expose feature_importances_ (e.g. it is a linear/SVM model).")
    importance_df = None

## 15. SHAP Analysis — Deeper, Per-Prediction Interpretability

Built-in feature importances only say *how much* a feature was used across all splits, not
*which direction* it pushes predictions (does a high value increase or decrease churn
risk?), nor how it interacts with other features. **SHAP (SHapley Additive exPlanations)**
values solve this: for every prediction, they attribute the gap between "average predicted
probability" and "this customer's predicted probability" fairly across the input features,
based on game-theoretic Shapley values.

We use `TreeExplainer`, which is exact and efficient for tree-ensemble models
(Random Forest / XGBoost / LightGBM), on a sample of the transformed test set.


In [ ]:
X_test_transformed = best_pipeline_preprocessor.transform(X_test)
X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=feature_names_transformed)

# Sample for speed if the test set is large; SHAP on tree models is fast, but this keeps
# the plotting step responsive regardless of dataset size.
sample_size = min(500, len(X_test_transformed_df))
X_shap_sample = X_test_transformed_df.sample(sample_size, random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(best_pipeline_classifier)
shap_values = explainer.shap_values(X_shap_sample)

# Some tree explainers return a list [class0, class1]; normalize to the churn-class values.
if isinstance(shap_values, list):
    shap_values_churn = shap_values[1]
else:
    shap_values_churn = shap_values

In [ ]:
shap.summary_plot(shap_values_churn, X_shap_sample, show=False)
plt.title("SHAP Summary — Impact of Each Feature on Churn Probability")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values_churn, X_shap_sample, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Mean |SHAP value|)")
plt.tight_layout()
plt.show()

**Cross-checking the drivers:** compare the top features from the built-in importance
plot (Section 14), the SHAP summary above, and the Mutual Information ranking from
Section 8. Features that show up as important across *all three* independent methods are
the most trustworthy churn drivers — that convergence is what we build the recommendations
in Section 17 on.


## 16. Winning Model Summary

Before moving to recommendations, we consolidate the result: the best-performing model
(`best_model_name`), its tuned decision threshold, and its held-out test metrics, computed
purely in-memory (this notebook does not persist anything to disk — see Section 1).


In [ ]:
summary = {
    "best_model": best_model_name,
    "chosen_threshold": float(best_threshold),
    "test_metrics": test_results_df.loc[best_model_name].to_dict(),
}
summary

## 17. Key Churn Factors & Business Recommendations

*(Read this section against the actual importance/SHAP plots once the notebook has been
run — the points below describe the standard, expected pattern for this dataset and should
be adjusted to match whatever the executed output actually shows.)*

### Likely top churn drivers, based on the EDA, statistical tests, feature importance and
SHAP analysis above:

1. **Complaints (`Complain`, `HighRiskComplaint`)** — customers who lodge a complaint,
   especially combined with low `SatisfactionScore`, churn at a much higher rate.
   **Recommendation:** route complaints to a fast-response resolution workflow with
   follow-up satisfaction checks; treat "complaint + low satisfaction" as an automatic
   trigger for proactive retention outreach.

2. **Tenure / `TenureGroup`** — newer customers (low tenure) are typically the highest-risk
   group; churn risk usually drops as customers become "established"/"loyal".
   **Recommendation:** invest in a structured onboarding journey for the first 1-2 months
   (welcome offers, usage tips, early check-ins) since this is where churn is most
   preventable and most costly per customer relative to lifetime value earned so far.

3. **Recency / Inactivity (`DaySinceLastOrder`, `InactivityFlag`)** — a long gap since the
   last order is a strong leading indicator, since disengagement precedes formal churn.
   **Recommendation:** set up automated re-engagement campaigns (personalized offers,
   reminders) triggered once a customer crosses the inactivity threshold identified here,
   rather than waiting for a fixed calendar cadence.

4. **Satisfaction Score** — low scores correlate strongly with churn even without a formal
   complaint being filed. **Recommendation:** treat low satisfaction as an early-warning
   signal on its own, not only when paired with a complaint; consider lightweight periodic
   satisfaction pulses for active customers.

5. **Cashback / Coupon behavior (`CashbackPerOrder`, `CouponUtilization`)** — patterns of
   reward usage relative to order volume may reveal price-sensitive segments who churn once
   promotions stop being competitive with alternatives.
   **Recommendation:** for customers whose retention depends heavily on cashback/coupons,
   consider tiered loyalty rewards that scale with tenure rather than one-off discounts, to
   build stickiness beyond price.

6. **Delivery distance (`WarehouseToHome`, `WarehouseDistanceTier`)** — customers far from
   a warehouse may experience slower/less reliable delivery.
   **Recommendation:** evaluate whether logistics/delivery SLAs for far-tier customers are
   materially worse, and prioritize warehouse network or courier-partner improvements in the
   highest-churn-risk delivery zones.

7. **City tier / Marital status / Number of addresses** — demographic and account-footprint
   features often show smaller, but still statistically significant, effects.
   **Recommendation:** use these as secondary segmentation variables for tailoring retention
   messaging (e.g. different offers by city tier), not as primary triggers on their own.

### Operationalizing the model

- **Score customers on a recurring schedule** (e.g. weekly) using the saved pipeline in
  `models/`, and route customers above the tuned probability threshold (Section 13) to the
  retention team's workflow.
- **Prioritize by expected value**, not just churn probability — combine the predicted churn
  probability with customer lifetime value so retention effort/budget goes to high-value,
  high-risk customers first.
- **Monitor for drift**: retrain periodically and re-validate the chosen threshold, since
  churn drivers can shift with seasonality, pricing changes, or new competitors.
- **A/B test interventions**: for each major driver above, run controlled experiments (e.g.
  proactive outreach for high-`HighRiskComplaint` customers vs. a holdout group) to confirm
  the recommendation actually reduces churn, rather than assuming correlation implies a
  fixable cause.


## 18. Conclusion & Limitations

**Summary of the pipeline:** raw Excel data was cleaned (standardized categories, median
imputation, outlier capping), explored to understand churn patterns, enriched with seven
domain-driven engineered features, checked for feature relevance via mutual information and
statistical tests, and used to train and compare eight classifier families under a
leakage-safe SMOTE pipeline. The strongest tree-based candidates (Random Forest, XGBoost,
LightGBM) were tuned via randomized search and evaluated on a held-out test set with
precision, recall, F1, ROC-AUC, PR-AUC and confusion matrices; the winning model's decision
threshold was further tuned, and its churn drivers were identified via built-in feature
importance and SHAP analysis.

**Limitations & future work:**
- The dataset is a static snapshot — it has no timestamps, so we cannot model *time-to-churn*
  or detect trends/seasonality. A survival-analysis approach (e.g. Cox proportional hazards)
  could complement this classification model if timestamped order history becomes available.
- SMOTE generates *synthetic* minority examples; results should be validated against a
  real-world holdout of actual churners over time (not just a random test split) before
  full production rollout.
- Feature importance/SHAP describe correlation with churn, not proven causation — the A/B
  testing step recommended in Section 17 is what would confirm a driver is actually
  actionable.
- Consider periodically re-tuning hyperparameters and re-checking the decision threshold as
  customer behavior and the product/market evolve, rather than treating this as a one-time
  model.
